# 1. Project Configuration

## 1.1 Central Configuration

Le projet repose sur une configuration centralisée implémentée dans :

```text
src/config.py
```

Ce composant constitue la source de référence utilisée par l'ensemble des scripts et notebooks du projet.

Son objectif est de garantir :

- la cohérence des chemins utilisés dans le projet ;
- la reproductibilité des traitements ;
- l'indépendance vis-à-vis de l'environnement d'exécution ;
- la centralisation des paramètres globaux de la plateforme.

---

## 1.2 Configuration Scope

Le fichier de configuration centralise notamment :

### Project Information

```text
Project Name
Version
Reporting Currency
```

### Analysis Parameters

```text
Start Date
End Date
Annualization Factor
```

### Documentation

```text
documentation/
```

### Data Storage

```text
data/01_raw
data/02_processed
data/03_universe
```

### Database

```text
database/
```

### Source Code

```text
src/
```

### Audit & Governance

```text
audit/source_registry.xlsx
audit/acquisition_log.xlsx
```

### Testing

```text
tests/acquisition
tests/quality
tests/portfolio
tests/risk
```

---

## 1.3 Usage

L'ensemble des scripts et notebooks du projet importe directement les paramètres et chemins définis dans :

```text
src/config.py
```

afin d'éviter l'utilisation de chemins relatifs codés en dur.

Cette approche garantit une gestion cohérente des ressources, facilite la maintenance du projet et simplifie les évolutions futures de l'architecture technique.

# 2. SRC-001 – Benchmark Dataset

## 2.1 Source

Provider :

```text
iShares / BlackRock
```

Dataset :

```text
Benchmark Holdings
```

Acquisition Script :

```text
src/acquisition/extract_benchmark.py
```

Dataset Produced :

```text
data/01_raw/benchmark/benchmark_holdings.xls
```

---

## 2.2 Technical Characteristics

Le benchmark est distribué sous la forme d'un document :

```text
SpreadsheetML XML
```

malgré son extension :

```text
.xls
```

Cette caractéristique a nécessité le développement d'un traitement spécifique pour l'extraction et la reconstruction des données.

---

## 2.3 Dataset Structure

Le benchmark contient les worksheets suivantes :

```text
Disclaimers
Holdings
Historical
Performance
Distributions
```

La worksheet :

```text
Holdings
```

constitue la principale source de données utilisée dans le projet, car elle contient la composition détaillée du portefeuille et les informations nécessaires à l'identification des instruments financiers.

---

## 2.4 Security Candidates Dataset

### 2.4.1 Objective

Identifier les instruments financiers présents dans le benchmark afin de préparer l'acquisition du Securities Dataset.

### 2.4.2 Implementation

Script :

```text
src/preparation/securities_candidates.py
```

Input :

```text
data/01_raw/benchmark/benchmark_holdings.xls
```

Output :

```text
data/01_raw/securities/security_candidates.csv
```

Le traitement :

1. extrait les données de la worksheet Holdings ;
2. reconstruit les enregistrements ;
3. sélectionne les positions Equity ;
4. consolide les instruments financiers.

### 2.4.3 Results

```text
120 positions Equity
120 instruments uniques
```

Le nombre d'instruments obtenu est cohérent avec le nombre de titres communiqué par le fournisseur dans les métadonnées du benchmark.

### 2.4.4 Selected Attributes

```text
Ticker
Name
Location
Exchange
Currency
Asset Class
```


# 3. SRC-002 – Securities Dataset

## 3.1 Objective

L'objectif du Securities Dataset est d'enrichir les instruments financiers identifiés dans le benchmark à l'aide des identifiants et métadonnées fournis par OpenFIGI.

Ce dataset constitue le Security Master utilisé par les phases ultérieures du projet.

---

## 3.2 Source

Provider :

```text
OpenFIGI
```

API :

```text
https://api.openfigi.com/v3/mapping
```

Acquisition Script :

```text
src/acquisition/extract_securities.py
```

Input :

```text
data/01_raw/securities/security_candidates.csv
```

Outputs :

```text
data/01_raw/securities/securities_master.csv

data/01_raw/securities/securities_unmatched.csv
```

---

## 3.3 Exchange Mapping

### 3.3.1 Objective

Les informations de cotation présentes dans le benchmark utilisent des libellés descriptifs qui ne sont pas directement exploitables par OpenFIGI.

Une table de correspondance a été construite afin de traduire les places de cotation du benchmark vers les codes d'échange attendus par OpenFIGI.

### 3.3.2 Implementation

Component :

```text
src/preparation/exchange_mapping.py
```

Le mapping couvre :

```text
29 places de cotation
```

identifiées dans le Security Candidates Dataset.

### 3.3.3 Validation

```text
29 exchanges dans le benchmark
29 exchanges couverts
0 exchange non mappé
```

La couverture du mapping a été validée avant l'exécution des requêtes OpenFIGI.

---

## 3.4 OpenFIGI Acquisition Process

Le traitement réalise les opérations suivantes :

1. lecture du Security Candidates Dataset ;
2. récupération du code OpenFIGI associé à chaque exchange ;
3. génération des requêtes OpenFIGI ;
4. récupération des instruments enrichis ;
5. gestion automatique des retries sur erreurs temporaires ;
6. construction du Security Master ;
7. isolation des instruments non enrichis.

Les erreurs HTTP temporaires suivantes sont automatiquement retraitées :

```text
429
500
502
503
504
```

---

## 3.5 Security Master Dataset

### Structure

Le Security Master contient notamment :

```text
ticker
figi
composite_figi
share_class_figi
security_name
security_type
security_type_2
market_sector
security_description
```

Ces informations sont associées aux attributs provenant du benchmark.

### Results

```text
120 instruments analysés
103 instruments enrichis
17 instruments non enrichis
85.83 % de couverture
```

Le fichier produit est :

```text
data/01_raw/securities/securities_master.csv
```

---

## 3.6 Unmatched Securities

Les instruments pour lesquels aucune correspondance OpenFIGI n'a été obtenue sont automatiquement isolés dans :

```text
data/01_raw/securities/securities_unmatched.csv
```

Cette séparation permet de poursuivre les analyses tout en conservant une traçabilité complète des exceptions.

Les titres non enrichis sont principalement concentrés sur certaines places de cotation locales et reflètent principalement des limitations liées aux conventions de ticker utilisées par certaines bourses.

---

# 4. Technical Logging

## 4.1 Objective

Le projet utilise un mécanisme de logging centralisé afin de journaliser les événements techniques produits par les scripts d'acquisition et de préparation.

---

## 4.2 Implementation

Component :

```text
src/utils/logger.py
```

Log File :

```text
logs/technical.log
```

Le Technical Log enregistre notamment :

- le démarrage des traitements ;
- les validations réalisées ;
- les avertissements ;
- les erreurs ;
- les informations d'exécution.

---

## 4.3 Log Rotation

Le logging repose sur un mécanisme :

```text
RotatingFileHandler
```

permettant :

- de limiter la taille des fichiers ;
- de conserver plusieurs historiques d'exécution ;
- d'éviter la croissance illimitée du log.

---

## 4.4 Technical Log vs Acquisition Log

Le projet distingue deux mécanismes complémentaires :

### Technical Log

```text
logs/technical.log
```

Objectif :

```text
Tracer l'exécution technique des traitements.
```

### Acquisition Log

```text
audit/acquisition_log.xlsx
```

Objectif :

```text
Tracer les acquisitions réalisées et les datasets produits.
```

Cette séparation permet de distinguer la traçabilité technique de la traçabilité métier.

# 5. Acquisition Governance

## 5.1 Objective

Le projet implémente un mécanisme de gouvernance dédié afin d'assurer la traçabilité complète des acquisitions réalisées tout au long du cycle de vie de la plateforme.

Chaque acquisition produit une entrée d'audit contenant les informations nécessaires au suivi, à la validation et à la reproductibilité des traitements.

---

## 5.2 Implementation

Component :

```text
src/acquisition/acquisition_logger.py
```

Audit File :

```text
audit/acquisition_log.xlsx
```

Le composant est utilisé par les scripts d'acquisition afin d'enregistrer automatiquement les métadonnées associées à chaque exécution.

---

## 5.3 Run Identification

Chaque acquisition se voit attribuer un identifiant unique :

```text
ACQ-YYYYMMDD-HHMMSS
```

Exemple :

```text
ACQ-20260907-164003
```

Cet identifiant permet d'assurer la traçabilité des traitements réalisés et facilite les investigations ultérieures.

---

## 5.4 Logged Attributes

Les informations suivantes sont enregistrées :

```text
Run ID
Acquisition Date
Source ID
Dataset
Provider
Period Covered
Output File
Storage Location
Status
Records Downloaded
Notes
```

---

## 5.5 Acquisition Status

Les acquisitions peuvent prendre les statuts suivants :

```text
Planned
Success
Partial Success
Failed
```

Ces statuts permettent un suivi standardisé de l'état des acquisitions.

---

## 5.6 Current Acquisition Results

À l'issue de la première exécution de la Phase 3 , les acquisitions suivantes ont été réalisées :

| Source ID | Dataset | Status |
|------------|------------|------------|
| SRC-001 | Benchmark Dataset | Success |
| SRC-002 | Securities Dataset | Partial Success |

Le statut :

```text
Partial Success
```

observé pour SRC-002 reflète la présence d'instruments non enrichis tout en confirmant le succès global de l'acquisition.

---

## 5.7 Governance Benefits

L'Acquisition Log permet notamment :

- la traçabilité des traitements ;
- le suivi des datasets produits ;
- la documentation des résultats d'acquisition ;
- la reproductibilité des exécutions ;
- l'audit des données collectées.

Ce mécanisme constitue la principale source de traçabilité métier de la plateforme.

# 6. Database Layer

## 6.1 Objective

Centraliser les datasets acquis dans une base SQLite
utilisée par les phases suivantes de la plateforme.

---

## 6.2 Database

Database File :

database/sdg_investment.db

---

## 6.3 Initial Tables
 
security_candidates  
 
securities_master  
 
securities_unmatched  

---

## 6.4 Purpose

La base SQLite constitue la source de référence utilisée par les phases suivantes du projet.

Les contrôles de qualité, la construction des univers et les analyses futures s'appuieront sur les données stockées dans la base plutôt que sur les fichiers CSV produits lors de l'acquisition.

# 7. Conclusion

La Phase 3 a permis la mise en place complète de la couche d'acquisition et de préparation des données.

Les objectifs suivants ont été atteints :

- acquisition automatisée du Benchmark Dataset (SRC-001) ;
- identification des instruments financiers présents dans le benchmark ;
- construction du Security Candidates Dataset ;
- mise en place du mapping des places de cotation ;
- acquisition et enrichissement OpenFIGI (SRC-002) ;
- construction du Security Master ;
- isolation des instruments non enrichis ;
- implémentation du Technical Log ;
- implémentation de l'Acquisition Log ;
- validation complète des traitements et des datasets produits.

La plateforme a volontairement été construite autour de deux sources initiales afin de valider l'architecture et les traitements sur un périmètre maîtrisé.

De nouvelles sources de données seront intégrées progressivement au cours des phases suivantes. 

Les datasets acquis ont été intégrés dans une base SQLite servant désormais de source de référence pour les phases suivantes de la plateforme.  